# YOLO26n ASFF — pipeline modular

Notebook ini memakai paket `skripsi`. Dataset v2, audit, evaluasi, dan protokol training sama untuk keempat arsitektur. **Preflight tidak melakukan training.** Training dan evaluasi hanya berjalan ketika flag sel masing-masing diaktifkan. Di Kaggle, aktifkan GPU dan Internet serta simpan `ROBOFLOW_API_KEY` dan (opsional) `WANDB_API_KEY` pada Secrets.


## 1. Bootstrap kode dan dependensi

Di Kaggle, kode di-clone dari repository publik. Setelah perubahan lokal dipush, buat versi notebook Kaggle baru agar checkout mengambil commit baru.


In [ ]:
from pathlib import Path
import subprocess
import sys

ON_KAGGLE = Path("/kaggle/working").is_dir()
REPO_URL = "https://github.com/lalalostcode/skripsi-deteksi-kendaraan-indonesia.git"
REPO_REF = "main"  # record the resolved commit below for reproducibility

if ON_KAGGLE:
    repo_root = Path("/kaggle/working/skripsi-code")
    if not (repo_root / ".git").exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(repo_root)], check=True)
    subprocess.run(["git", "-C", str(repo_root), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(repo_root), "checkout", "--detach", "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "ultralytics==8.4.155", "wandb==0.28.1", "roboflow==1.4.0",
                    "pycocotools>=2.0.10", "thop", "pyyaml", "pandas", "pillow", "matplotlib"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_root), "--no-deps"], check=True)
else:
    repo_root = Path.cwd().resolve()
    for candidate in (repo_root, *repo_root.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "skripsi").is_dir():
            repo_root = candidate
            break
    else:
        raise FileNotFoundError("Buka notebook dari checkout repository skripsi")
    # `uv sync` environments may not ship pip; imports work directly from src.
    sys.path.insert(0, str(repo_root / "src"))

git_commit = subprocess.check_output(["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True).strip()
print("Repository:", repo_root)
print("Commit:", git_commit)


## 2. Konfigurasi eksperimen

Gunakan satu seed per run Kaggle. Ubah hanya `active_seed` untuk run berikutnya.


In [ ]:
from skripsi import ExperimentConfig, preflight, run_training, evaluate_checkpoint

architecture = "asff"
active_seed = 42  # ganti menjadi 123 atau 2026 untuk seed berikutnya
local_v2_candidates = (
    repo_root / "data" / "raw" / "poltrada-original-raw-2-v2",
    repo_root / "data" / "audited_v2",
)
dataset_dir = next((path for path in local_v2_candidates if path.is_dir()), None) if not ON_KAGGLE else None
output_root = Path("/kaggle/working/yolo26n_research") if ON_KAGGLE else repo_root / "outputs" / "research"

config = ExperimentConfig(
    architecture=architecture,
    seed=active_seed,
    dataset_dir=dataset_dir,
    output_root=output_root,
    git_commit=git_commit,
)
print(config)


## 3. Preflight tanpa training

Salin atau unduh dataset v2 ke area kerja, konversi polygon pada salinan, audit fingerprint, dan periksa graph dengan satu forward sintetis. Pada lokal tanpa dataset v2, tambahkan `dataset_dir` atau `ROBOFLOW_API_KEY`.


In [ ]:
dataset = preflight(config)
print('Fingerprint:', dataset.fingerprint)
print('Runtime YAML:', dataset.runtime_yaml)


## 4. Training — jalankan hanya bila Anda menghendaki

Pemanggilan ini disiapkan untuk pengguna; default `False` sehingga Run All tidak memulai training.


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    training_result = run_training(config, dataset)
    print(training_result)


## 5. Evaluasi checkpoint — tanpa training

Setelah `best.pt` tersedia, aktifkan sel ini. Validation memilih ambang confidence; test menghasilkan tabel 3.5–3.8.


In [ ]:
RUN_EVALUATION = False
checkpoint = config.run_dir / 'weights' / 'best.pt'
if RUN_EVALUATION:
    evaluation_result = evaluate_checkpoint(config, checkpoint, dataset)
    print(evaluation_result['overall'])
    print('Hasil:', evaluation_result['evaluation_dir'])
